# ViT-VAE Training Notebook

**Before running:**
1. `Runtime → Change runtime type → T4 GPU`
2. Run all cells top to bottom (Runtime → Run all)

**What this notebook does:**
- Clones the project repo from GitHub (all model/training code lives there)
- Mounts Google Drive to save the checkpoint across sessions
- Downloads Fashion-MNIST automatically via torchvision
- Trains the ViT-VAE for 30 epochs (~25 min on T4)
- Saves the best checkpoint to `/content/drive/MyDrive/vit_vae_best.pth`
- Shows a quick reconstruction preview so you can verify quality

## 1. Mount Google Drive
This keeps the checkpoint alive after the Colab session ends.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted at /content/drive')

## 2. Clone the Repository
Replace the URL with your own GitHub repo once you push the code.

In [ ]:
# ── Your GitHub repo URL ─────────────────────────────────────────────────────
REPO_URL = 'https://github.com/oktay-us/vit-vae-explorer.git'
# ─────────────────────────────────────────────────────────────────────────────

import os

if not os.path.exists('/content/vit-vae-explorer'):
    !git clone {REPO_URL} /content/vit-vae-explorer
else:
    # Pull latest changes if the repo was already cloned in a previous session.
    !git -C /content/vit-vae-explorer pull

# Add repo root to Python path so we can do `from src.model import ViTVAE`.
import sys
sys.path.insert(0, '/content/vit-vae-explorer')
os.chdir('/content/vit-vae-explorer')

print('Repo ready. Working directory:', os.getcwd())

## 3. Install Extra Packages
Colab has PyTorch + CUDA pre-installed. We only need a few extras.

In [ ]:
!pip install einops -q
print('Extra packages installed.')

## 4. Verify GPU
Make sure the T4 GPU is available before starting training.

In [ ]:
import torch

print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU             : {torch.cuda.get_device_name(0)}')
    print(f'VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('No GPU detected — go to Runtime → Change runtime type → T4 GPU')

## 5. Verify Model Architecture
Run a dummy forward pass to confirm shapes and parameter count before committing to 30 epochs.

In [ ]:
from src.model import ViTVAE

model = ViTVAE()

# Dummy forward pass: batch of 2 images
dummy = torch.randn(2, 1, 28, 28)
x_recon, mu, log_var = model(dummy)

print(f'Input shape      : {dummy.shape}')    # (2, 1, 28, 28)
print(f'Reconstruction   : {x_recon.shape}')  # (2, 1, 28, 28)
print(f'mu shape         : {mu.shape}')        # (2, 32)
print(f'log_var shape    : {log_var.shape}')   # (2, 32)

total_params = sum(p.numel() for p in model.parameters())
print(f'\nTotal parameters : {total_params:,}')  # ~3M

## 6. Train
This cell trains for 30 epochs and saves the best checkpoint to Google Drive.
Expected time: ~25 minutes on a Colab T4 GPU.

In [ ]:
from src.train import train

# The checkpoint is saved to Google Drive so it survives session disconnects.
CHECKPOINT_PATH = '/content/drive/MyDrive/vit_vae_best.pth'

trained_model = train(
    model=None,               # creates a fresh ViTVAE internally
    data_dir='./data',        # Fashion-MNIST downloaded here
    epochs=30,
    batch_size=128,
    lr=1e-3,
    beta=0.5,                 # KL weight (β-VAE)
    save_path=CHECKPOINT_PATH,
    num_workers=2,
)

## 7. Quick Visual Check
Display 8 test images alongside their reconstructions to sanity-check quality.

In [ ]:
import matplotlib.pyplot as plt
from torchvision import datasets, transforms

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
trained_model = trained_model.to(device).eval()

test_dataset = datasets.FashionMNIST(
    root='./data', train=False, download=True,
    transform=transforms.ToTensor()
)

# Pick 8 test images
n = 8
imgs = torch.stack([test_dataset[i][0] for i in range(n)])  # (8, 1, 28, 28)

with torch.no_grad():
    imgs_gpu = imgs.to(device)
    recons, _, _ = trained_model(imgs_gpu)   # (8, 1, 28, 28)
    recons = recons.cpu()

# Plot: top row = originals, bottom row = reconstructions
fig, axes = plt.subplots(2, n, figsize=(n * 1.5, 3))
for i in range(n):
    axes[0, i].imshow(imgs[i, 0].numpy(), cmap='gray')
    axes[0, i].axis('off')
    axes[1, i].imshow(recons[i, 0].numpy(), cmap='gray')
    axes[1, i].axis('off')

axes[0, 0].set_title('Original', loc='left', fontsize=9)
axes[1, 0].set_title('Reconstructed', loc='left', fontsize=9)
plt.suptitle('ViT-VAE Reconstructions on Fashion-MNIST Test Set', y=1.02)
plt.tight_layout()
plt.show()

print(f'\nCheckpoint saved to: {CHECKPOINT_PATH}')
print('Download it from Google Drive and place it at checkpoints/best_model.pth in your local repo.')